<h4> STADIOalot Return Prediction</h4>
<h4>Notebook 1 — Data Preprocessing</h4>

<h4>Purpose </h4>
This notebook prepares the Synthetic Dataset for E-Commerce Return Analysis for machine-learning modelling. The preprocessing stage performs:

- Data loading
- Data inspection
- Column-name standardisation
- Duplicate detection and removal
- Missing-value inspection
- Target-variable preparation
- Data-type conversion
- Identification and removal of target leakage
- Basic data-quality checks
- Export of the cleaned dataset

<h4> Target variable</h4>
The target variable is: `Return_Status`
The classification problem is:

- `Returned` = 1
- `Not Returned` = 0

<h4> Leakage prevention </h4>
The objective is to predict whether an order will be returned before the return occurs. Therefore, variables that are only known after the return process, such as:

- Return_Date
- Return_Reason
- Days_to_Return
must not be used as predictive features.

<h4> Input </h4>
`data/raw/ecommerce_returns_synthetic_data.csv`

<h4> Output </h4>
`data/processed/returns_preprocessed.csv`

In [1]:
# ============================================================
# 1. IMPORT LIBRARIES AND CONFIGURE DISPLAY
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
from IPython.display import display
import pandas as pd


# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.3f}".format)


print("=" * 70)
print("STADIOalot RETURN PREDICTION — DATA PREPROCESSING")
print("=" * 70)
print("Libraries loaded successfully.")

# ============================================================
# 2. DEFINE PROJECT PATHS
# ============================================================

# Allow Pandas to display the complete contents of each cell
pd.set_option("display.max_colwidth", None)


# Identify the project root
ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

ROOT = ROOT.resolve()


# Define full absolute paths
RAW_PATH = (
    ROOT
    / "data"
    / "raw"
    / "ecommerce_returns_synthetic_data.csv"
).resolve()

OUTPUT_PATH = (
    ROOT
    / "data"
    / "processed"
    / "returns_preprocessed.csv"
).resolve()


# Create output directory if it does not exist
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)


# Create path table
paths_table = pd.DataFrame({
    "Item": [
        "Project Root",
        "Raw Dataset",
        "Processed Dataset"
    ],
    "Full Path": [
        str(ROOT),
        str(RAW_PATH),
        str(OUTPUT_PATH)
    ],
    "Status": [
        "FOUND" if ROOT.exists() else "NOT FOUND",
        "FOUND" if RAW_PATH.exists() else "NOT FOUND",
        "READY" if OUTPUT_PATH.parent.exists() else "NOT READY"
    ]
})


# Display the complete table
display(paths_table)

STADIOalot RETURN PREDICTION — DATA PREPROCESSING
Libraries loaded successfully.


,Item,Full Path,Status
0,Project Root,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction,FOUND
1,Raw Dataset,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ecommerce_returns_synthetic_data.csv,FOUND
2,Processed Dataset,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\processed\returns_preprocessed.csv,READY


In [2]:
# ============================================================
# 3. LOAD DATASET
# ============================================================

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Dataset could not be found at:\n{RAW_PATH}"
    )

df = pd.read_csv(RAW_PATH)


# Dataset summary table
dataset_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Memory Usage (MB)"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2)
    ]
})


print("Dataset successfully loaded.")
display(dataset_summary)

# ============================================================
# 4. DATASET PREVIEW
# ============================================================

print("First five records:")
display(df.head())

print("Last five records:")
display(df.tail())

Dataset successfully loaded.


,Metric,Value
0,Rows,10000.000
1,Columns,17.000
2,Memory Usage (MB),6.780


First five records:


,Order_ID,Product_ID,User_ID,Order_Date,Return_Date,Product_Category,Product_Price,Order_Quantity,Return_Reason,Return_Status,Days_to_Return,User_Age,User_Gender,User_Location,Payment_Method,Shipping_Method,Discount_Applied
0,ORD00000000,PROD00000000,USER00000000,2023-08-05,2024-08-26,Clothing,411.590,3,Changed mind,Returned,387.000,58,Male,City54,Debit Card,Next-Day,45.270
1,ORD00000001,PROD00000001,USER00000001,2023-10-09,2023-11-09,Books,288.880,3,Wrong item,Returned,31.000,68,Female,City85,Credit Card,Express,47.790
2,ORD00000002,PROD00000002,USER00000002,2023-05-06,NaN,Toys,390.030,5,NaN,Not Returned,NaN,22,Female,City30,Debit Card,Next-Day,26.640
3,ORD00000003,PROD00000003,USER00000003,2024-08-29,NaN,Toys,401.090,3,NaN,Not Returned,NaN,40,Male,City95,PayPal,Next-Day,15.370
4,ORD00000004,PROD00000004,USER00000004,2023-01-16,NaN,Books,110.090,4,NaN,Not Returned,NaN,34,Female,City80,Gift Card,Standard,16.370


Last five records:


,Order_ID,Product_ID,User_ID,Order_Date,Return_Date,Product_Category,Product_Price,Order_Quantity,Return_Reason,Return_Status,Days_to_Return,User_Age,User_Gender,User_Location,Payment_Method,Shipping_Method,Discount_Applied
9995,ORD00009995,PROD00009995,USER00009995,2023-10-20,NaN,Home,142.500,4,NaN,Not Returned,NaN,37,Male,City40,PayPal,Standard,34.270
9996,ORD00009996,PROD00009996,USER00009996,2023-02-25,NaN,Electronics,484.630,3,NaN,Not Returned,NaN,69,Male,City62,Debit Card,Express,25.440
9997,ORD00009997,PROD00009997,USER00009997,2024-05-10,NaN,Toys,386.570,5,NaN,Not Returned,NaN,46,Male,City74,Credit Card,Next-Day,12.670
9998,ORD00009998,PROD00009998,USER00009998,2024-02-13,NaN,Toys,129.220,1,NaN,Not Returned,NaN,34,Female,City34,Gift Card,Express,49.970
9999,ORD00009999,PROD00009999,USER00009999,2024-12-08,2024-09-09,Toys,459.420,5,Not as described,Returned,-90.000,24,Female,City51,Debit Card,Express,16.050


In [3]:
# ============================================================
# 5. DATASET STRUCTURE AND MISSING-VALUE REPORT
# ============================================================

structure_table = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-Null Values": df.notna().sum().values,
    "Missing Values": df.isna().sum().values,
    "Missing %": (df.isna().sum() / len(df) * 100).values,
    "Unique Values": df.nunique(dropna=True).values
})

# Sort columns by number of missing values
structure_table = (
    structure_table
    .sort_values("Missing Values", ascending=False)
    .reset_index(drop=True)
)

print("Dataset Structure and Missing-Value Report:")
display(structure_table)

# ============================================================
# 6. DUPLICATE REPORT
# ============================================================

duplicate_count = df.duplicated().sum()

duplicate_report = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Duplicate Rows",
        "Unique Rows",
        "Duplicate Percentage"
    ],
    "Value": [
        len(df),
        duplicate_count,
        len(df) - duplicate_count,
        round((duplicate_count / len(df)) * 100, 2)
    ]
})
print("\nDuplicate Record Report: ")
display(duplicate_report)

Dataset Structure and Missing-Value Report:


,Column,Data Type,Non-Null Values,Missing Values,Missing %,Unique Values
0,Return_Date,str,5052,4948,49.480,729
1,Days_to_Return,float64,5052,4948,49.480,1249
2,Return_Reason,str,5052,4948,49.480,4
3,Order_ID,str,10000,0,0.000,10000
4,User_ID,str,10000,0,0.000,10000
5,Order_Date,str,10000,0,0.000,731
6,Product_Category,str,10000,0,0.000,5
7,Product_Price,float64,10000,0,0.000,9074
8,Product_ID,str,10000,0,0.000,10000
9,Order_Quantity,int64,10000,0,0.000,5



Duplicate Record Report: 


,Metric,Value
0,Total Rows,10000.000
1,Duplicate Rows,0.000
2,Unique Rows,10000.000
3,Duplicate Percentage,0.000


In [4]:
# ============================================================
# 7. STANDARDISE COLUMN NAMES
# ============================================================

original_columns = df.columns.tolist()

df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

standardised_columns = df.columns.tolist()


column_name_report = pd.DataFrame({
    "Original Column": original_columns,
    "Standardised Column": standardised_columns
})

display(column_name_report)

,Original Column,Standardised Column
0,Order_ID,Order_ID
1,Product_ID,Product_ID
2,User_ID,User_ID
3,Order_Date,Order_Date
4,Return_Date,Return_Date
5,Product_Category,Product_Category
6,Product_Price,Product_Price
7,Order_Quantity,Order_Quantity
8,Return_Reason,Return_Reason
9,Return_Status,Return_Status


In [5]:
# ============================================================
# 8. IDENTIFY TARGET VARIABLE
# ============================================================

target_candidates = [
    "Return_Status",
    "return_status",
    "ReturnStatus"
]

target_column = None

for column in target_candidates:
    if column in df.columns:
        target_column = column
        break


if target_column is None:
    raise ValueError(
        "Return_Status could not be found.\n"
        f"Available columns: {df.columns.tolist()}"
    )


print(f"Target column identified: {target_column}")


target_values = (
    df[target_column]
    .value_counts(dropna=False)
    .reset_index()
)

target_values.columns = [
    "Original Target Value",
    "Frequency"
]

target_values["Percentage"] = (
    target_values["Frequency"]
    / len(df)
    * 100
)
print("\nTarget Value Table")
display(target_values.style.hide(axis="index"))

# ============================================================
# 9. STANDARDISE TARGET LABELS
# ============================================================

df[target_column] = (
    df[target_column]
    .astype(str)
    .str.strip()
    .str.lower()
)


target_mapping = {
    "returned": 1,
    "return": 1,
    "yes": 1,
    "1": 1,

    "not returned": 0,
    "not_returned": 0,
    "no": 0,
    "0": 0
}


df["Return_Status"] = (
    df[target_column]
    .map(target_mapping)
)


# Identify values that could not be mapped
unmapped = (
    df.loc[
        df["Return_Status"].isna(),
        target_column
    ]
    .unique()
)


if len(unmapped) > 0:
    raise ValueError(
        f"Unrecognised target values: {unmapped}"
    )


# Remove original target column if necessary
if target_column != "Return_Status":
    df = df.drop(columns=[target_column])


# Convert target to integer
df["Return_Status"] = df["Return_Status"].astype(int)


print("Target variable successfully encoded.")
print("Returned = 1")
print("Not Returned = 0")

# ============================================================
# 10. TARGET DISTRIBUTION
# ============================================================

target_distribution = (
    df["Return_Status"]
    .value_counts()
    .sort_index()
    .reset_index()
)

target_distribution.columns = [
    "Return Status",
    "Frequency"
]

target_distribution["Percentage"] = (
    target_distribution["Frequency"]
    / len(df)
    * 100
)

target_distribution["Label"] = (
    target_distribution["Return Status"]
    .map({
        0: "Not Returned",
        1: "Returned"
    })
)

target_distribution = target_distribution[
    [
        "Return Status",
        "Label",
        "Frequency",
        "Percentage"
    ]
]

# Display table without the pandas index
display(target_distribution.style.hide(axis="index"))

Target column identified: Return_Status

Target Value Table


Original Target Value,Frequency,Percentage
Returned,5052,50.520000
Not Returned,4948,49.480000


Target variable successfully encoded.
Returned = 1
Not Returned = 0


Return Status,Label,Frequency,Percentage
0,Not Returned,4948,49.480000
1,Returned,5052,50.520000


In [6]:
# ============================================================
# 11. REMOVE DUPLICATE ROWS
# ============================================================

duplicates_before = df.duplicated().sum()

rows_before = len(df)

df = (
    df
    .drop_duplicates()
    .reset_index(drop=True)
)

duplicates_after = df.duplicated().sum()

rows_after = len(df)


duplicate_removal_report = pd.DataFrame({
    "Metric": [
        "Rows Before Removal",
        "Duplicate Rows Removed",
        "Rows After Removal",
        "Remaining Duplicate Rows"
    ],
    "Value": [
        rows_before,
        duplicates_before,
        rows_after,
        duplicates_after
    ]
})


display(duplicate_removal_report)

,Metric,Value
0,Rows Before Removal,10000
1,Duplicate Rows Removed,0
2,Rows After Removal,10000
3,Remaining Duplicate Rows,0


In [7]:
# ============================================================
# 12. IDENTIFY TARGET LEAKAGE
# ============================================================

possible_leakage_columns = [
    "Return_Date",
    "return_date",
    "Return_Reason",
    "return_reason",
    "Days_to_Return",
    "days_to_return",
    "Refund_Date",
    "refund_date",
    "Refund_Amount",
    "refund_amount"
]


leakage_columns = [
    column
    for column in possible_leakage_columns
    if column in df.columns
]


if leakage_columns:

    leakage_report = pd.DataFrame({
        "Leakage Column": leakage_columns,
        "Reason for Removal": [
            "Information available only after the return process"
            for _ in leakage_columns
        ]
    })

else:

    leakage_report = pd.DataFrame({
        "Leakage Column": ["None detected"],
        "Reason for Removal": [
            "No post-return variables were present"
        ]
    })

print("Leakage Report")
display(leakage_report)

# ============================================================
# 13. REMOVE TARGET-LEAKAGE VARIABLES
# ============================================================

columns_before = df.shape[1]

df = df.drop(
    columns=leakage_columns,
    errors="ignore"
)

columns_after = df.shape[1]


leakage_removal_summary = pd.DataFrame({
    "Metric": [
        "Columns Before Leakage Removal",
        "Leakage Columns Removed",
        "Columns After Leakage Removal"
    ],
    "Value": [
        columns_before,
        len(leakage_columns),
        columns_after
    ]
})

print("Leakage Removal Summary")
display(leakage_removal_summary)

# ============================================================
# 14. REMAINING FEATURES
# ============================================================

remaining_columns = pd.DataFrame({
    "Column Number": range(1, len(df.columns) + 1),
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values
})

print("Remaining Columns After Removal")
display(remaining_columns)

Leakage Report


,Leakage Column,Reason for Removal
0,Return_Date,Information available only after the return process
1,Return_Reason,Information available only after the return process
2,Days_to_Return,Information available only after the return process


Leakage Removal Summary


,Metric,Value
0,Columns Before Leakage Removal,17
1,Leakage Columns Removed,3
2,Columns After Leakage Removal,14


Remaining Columns After Removal


,Column Number,Column,Data Type
0,1,Order_ID,str
1,2,Product_ID,str
2,3,User_ID,str
3,4,Order_Date,str
4,5,Product_Category,str
5,6,Product_Price,float64
6,7,Order_Quantity,int64
7,8,Return_Status,int64
8,9,User_Age,int64
9,10,User_Gender,str


In [8]:
# ============================================================
# 15. CONVERT DATE COLUMNS
# ============================================================

date_columns = []
date_conversion_report = []


for column in df.columns:

    if column == "Return_Status":
        continue

    if "date" in column.lower():

        converted = pd.to_datetime(
            df[column],
            errors="coerce"
        )

        valid_percentage = converted.notna().mean()

        if valid_percentage >= 0.70:

            df[column] = converted

            date_columns.append(column)

            date_conversion_report.append({
                "Column": column,
                "Valid Dates (%)": round(
                    valid_percentage * 100,
                    2
                ),
                "Conversion": "Converted"
            })


if date_conversion_report:

    date_conversion_report = pd.DataFrame(
        date_conversion_report
    )

else:

    date_conversion_report = pd.DataFrame({
        "Column": ["None detected"],
        "Valid Dates (%)": [np.nan],
        "Conversion": ["No suitable date columns found"]
    })

print("Date Conversion Report")
display(date_conversion_report)

# ============================================================
# 16. DATA TYPES AFTER DATE CONVERSION
# ============================================================

dtype_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-Null Values": df.notna().sum().values,
    "Missing Values": df.isna().sum().values
})

print("Data Type Report After Conversion")
display(dtype_report)

Date Conversion Report


,Column,Valid Dates (%),Conversion
0,Order_Date,100.000,Converted


Data Type Report After Conversion


,Column,Data Type,Non-Null Values,Missing Values
0,Order_ID,str,10000,0
1,Product_ID,str,10000,0
2,User_ID,str,10000,0
3,Order_Date,datetime64[us],10000,0
4,Product_Category,str,10000,0
5,Product_Price,float64,10000,0
6,Order_Quantity,int64,10000,0
7,Return_Status,int64,10000,0
8,User_Age,int64,10000,0
9,User_Gender,str,10000,0


In [9]:
# ============================================================
# 17. NUMERIC DATA QUALITY CHECKS
# ============================================================

numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()


if "Return_Status" in numeric_columns:
    numeric_columns.remove("Return_Status")


quality_report = []


for column in numeric_columns:

    column_lower = column.lower()

    if any(
        keyword in column_lower
        for keyword in [
            "price",
            "amount",
            "quantity",
            "discount"
        ]
    ):

        numeric_values = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        negative_count = (
            numeric_values < 0
        ).sum()

        quality_report.append({
            "Column": column,
            "Negative Values": negative_count,
            "Action": (
                "Converted to missing values"
                if negative_count > 0
                else "No action required"
            )
        })


if quality_report:

    quality_report = pd.DataFrame(
        quality_report
    )

else:

    quality_report = pd.DataFrame({
        "Column": ["No relevant numeric columns detected"],
        "Negative Values": [0],
        "Action": ["No action required"]
    })

print("Numeric Data Quality Report")
display(quality_report)

# ============================================================
# 18. HANDLE INVALID NEGATIVE NUMERIC VALUES
# ============================================================

corrections = []


for column in numeric_columns:

    column_lower = column.lower()

    if any(
        keyword in column_lower
        for keyword in [
            "price",
            "amount",
            "quantity",
            "discount"
        ]
    ):

        numeric_values = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        negative_mask = numeric_values < 0
        negative_count = negative_mask.sum()


        if negative_count > 0:

            df.loc[
                negative_mask,
                column
            ] = np.nan


        corrections.append({
            "Column": column,
            "Invalid Negative Values": negative_count,
            "Values Converted to Missing": negative_count
        })


if corrections:

    corrections = pd.DataFrame(corrections)

else:

    corrections = pd.DataFrame({
        "Column": ["None"],
        "Invalid Negative Values": [0],
        "Values Converted to Missing": [0]
    })

print("Corrections to Numeric Values")
display(corrections)

Numeric Data Quality Report


,Column,Negative Values,Action
0,Product_Price,0,No action required
1,Order_Quantity,0,No action required
2,Discount_Applied,0,No action required


Corrections to Numeric Values


,Column,Invalid Negative Values,Values Converted to Missing
0,Product_Price,0,0
1,Order_Quantity,0,0
2,Discount_Applied,0,0


In [10]:
# ============================================================
# 19. REMOVE ROWS WITH MISSING TARGET
# ============================================================

rows_before = len(df)

missing_target_count = (
    df["Return_Status"]
    .isna()
    .sum()
)


df = (
    df
    .dropna(subset=["Return_Status"])
    .reset_index(drop=True)
)


rows_after = len(df)


target_removal_report = pd.DataFrame({
    "Metric": [
        "Rows Before Removal",
        "Rows With Missing Target",
        "Rows Removed",
        "Rows After Removal"
    ],
    "Value": [
        rows_before,
        missing_target_count,
        rows_before - rows_after,
        rows_after
    ]
})

print("Remove Rows with Missing Target")
display(target_removal_report)

# ============================================================
# 20. FINAL MISSING-VALUE REPORT
# ============================================================

final_missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing Values": df.isna().sum().values
})

final_missing_report["Missing %"] = (
    final_missing_report["Missing Values"]
    / len(df)
    * 100
)

final_missing_report = (
    final_missing_report
    .sort_values(
        "Missing Values",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Missing Target Report")
display(final_missing_report)

Remove Rows with Missing Target


,Metric,Value
0,Rows Before Removal,10000
1,Rows With Missing Target,0
2,Rows Removed,0
3,Rows After Removal,10000


Missing Target Report


,Column,Missing Values,Missing %
0,Order_ID,0,0.000
1,Product_ID,0,0.000
2,User_ID,0,0.000
3,Order_Date,0,0.000
4,Product_Category,0,0.000
5,Product_Price,0,0.000
6,Order_Quantity,0,0.000
7,Return_Status,0,0.000
8,User_Age,0,0.000
9,User_Gender,0,0.000


In [11]:
# ============================================================
# 21. FINAL PREPROCESSING SUMMARY
# ============================================================

final_summary = pd.DataFrame({
    "Metric": [
        "Final Rows",
        "Final Columns",
        "Duplicate Rows",
        "Missing Target Values",
        "Target Variable",
        "Date Columns Converted",
        "Leakage Columns Removed"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        df.duplicated().sum(),
        df["Return_Status"].isna().sum(),
        "Return_Status",
        len(date_columns),
        len(leakage_columns)
    ]
})

print("Processing Summary")
display(final_summary)

# ============================================================
# 22. FINAL PREPROCESSED DATASET PREVIEW
# ============================================================

print("Final preprocessed dataset:")
display(df.head(10))

Processing Summary


,Metric,Value
0,Final Rows,10000
1,Final Columns,14
2,Duplicate Rows,0
3,Missing Target Values,0
4,Target Variable,Return_Status
5,Date Columns Converted,1
6,Leakage Columns Removed,3


Final preprocessed dataset:


,Order_ID,Product_ID,User_ID,Order_Date,Product_Category,Product_Price,Order_Quantity,Return_Status,User_Age,User_Gender,User_Location,Payment_Method,Shipping_Method,Discount_Applied
0,ORD00000000,PROD00000000,USER00000000,2023-08-05,Clothing,411.590,3,1,58,Male,City54,Debit Card,Next-Day,45.270
1,ORD00000001,PROD00000001,USER00000001,2023-10-09,Books,288.880,3,1,68,Female,City85,Credit Card,Express,47.790
2,ORD00000002,PROD00000002,USER00000002,2023-05-06,Toys,390.030,5,0,22,Female,City30,Debit Card,Next-Day,26.640
3,ORD00000003,PROD00000003,USER00000003,2024-08-29,Toys,401.090,3,0,40,Male,City95,PayPal,Next-Day,15.370
4,ORD00000004,PROD00000004,USER00000004,2023-01-16,Books,110.090,4,0,34,Female,City80,Gift Card,Standard,16.370
5,ORD00000005,PROD00000005,USER00000005,2024-02-14,Electronics,252.120,1,1,46,Female,City47,Debit Card,Next-Day,47.610
6,ORD00000006,PROD00000006,USER00000006,2023-05-29,Clothing,382.890,2,1,25,Male,City50,Credit Card,Express,28.490
7,ORD00000007,PROD00000007,USER00000007,2023-02-09,Electronics,306.390,3,1,67,Female,City39,Credit Card,Express,38.910
8,ORD00000008,PROD00000008,USER00000008,2024-12-25,Home,302.400,5,1,54,Male,City81,Debit Card,Standard,18.500
9,ORD00000009,PROD00000009,USER00000009,2023-03-10,Clothing,294.940,3,1,44,Male,City82,Credit Card,Standard,45.210


In [12]:
# ============================================================
# 23. SAVE PREPROCESSED DATASET
# ============================================================

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)


df.to_csv(
    OUTPUT_PATH,
    index=False
)


save_report = pd.DataFrame({
    "Item": [
        "Output File",
        "Rows Saved",
        "Columns Saved",
        "File Exists"
    ],
    "Value": [
        str(OUTPUT_PATH),
        len(df),
        len(df.columns),
        OUTPUT_PATH.exists()
    ]
})


display(save_report)

print("\nPreprocessed dataset successfully saved.")

,Item,Value
0,Output File,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\processed\returns_preprocessed.csv
1,Rows Saved,10000
2,Columns Saved,14
3,File Exists,True



Preprocessed dataset successfully saved.


In [13]:
# ============================================================
# 24. FINAL VALIDATION
# ============================================================

validation_checks = pd.DataFrame({
    "Validation Check": [
        "Dataset contains rows",
        "Target column exists",
        "Target contains only 0 and 1",
        "No duplicate rows",
        "No missing target values",
        "Processed file exists"
    ],
    "Result": [
        len(df) > 0,
        "Return_Status" in df.columns,
        df["Return_Status"].isin([0, 1]).all(),
        df.duplicated().sum() == 0,
        df["Return_Status"].isna().sum() == 0,
        OUTPUT_PATH.exists()
    ]
})


validation_checks["Status"] = np.where(
    validation_checks["Result"],
    "PASS",
    "CHECK"
)

print("Validation Check")
display(validation_checks)

# ============================================================
# 25. PREPROCESSING SUMMARY
# ============================================================

preprocessing_summary = pd.DataFrame({
    "Item": [
        "Final Dataset Shape",
        "Target Variable",
        "Processed Dataset"
    ],
    "Value": [
        str(df.shape),
        "Return_Status",
        OUTPUT_PATH
    ]
})
print("Preprocessing Summary")
display(preprocessing_summary)

Validation Check


,Validation Check,Result,Status
0,Dataset contains rows,True,PASS
1,Target column exists,True,PASS
2,Target contains only 0 and 1,True,PASS
3,No duplicate rows,True,PASS
4,No missing target values,True,PASS
5,Processed file exists,True,PASS


Preprocessing Summary


,Item,Value
0,Final Dataset Shape,"(10000, 14)"
1,Target Variable,Return_Status
2,Processed Dataset,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\processed\returns_preprocessed.csv
